# Phase 5 (LSTM): RUL via independent LSTM branch, with hyperparameter search
TCN encoder + Detection/Diagnosis/Severity load from Phase 4 and stay frozen. RUL gets its own independent LSTM branch (zero shared weights). Searches units/num_layers/dropout/learning_rate via Hyperband before the final full-size training run. Includes the checkpoint safeguard from the earlier overwrite bug.

In [1]:
import sys, os, json
sys.path.insert(0, "/Users/harsh/Documents/UAV_Engine/validation")
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tf_data_pipeline import make_dataset
from model_architectures import (build_encoder_input, build_encoder, build_detection_head,
                                   build_diagnosis_head, build_severity_head, build_rul_head,
                                   build_rul_lstm_encoder)

tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
DATA_DIR = "/Users/harsh/Documents/UAV_Engine/validation/chunks"
INDEX_PATH = os.path.join(DATA_DIR, "scenario_index.json")
SCALER_PATH = "/Users/harsh/Documents/UAV_Engine/validation/models/scaler.pkl"
PHASE4_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase4_severity.keras"
PHASE5_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_lstm.keras"
TEMP_CKPT = PHASE5_CKPT.replace(".keras", "_temp.keras")
BEST_MAE_RECORD = "/Users/harsh/Documents/UAV_Engine/validation/models/phase5_lstm_best_mae.json"
CLASS_WEIGHTS_PATH = "/Users/harsh/Documents/UAV_Engine/validation/models/diag_class_weights_keras.npy"

BATCH_SIZE = 128
STEPS_PER_EPOCH = 4196
VALIDATION_STEPS = 522
TEST_STEPS = 522
EPOCHS = 12

In [3]:
class_weights_np = np.load(CLASS_WEIGHTS_PATH)
class_weights_tf = tf.constant(class_weights_np, dtype=tf.float32)

def weighted_diagnosis_loss(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    bsz = tf.shape(y_true_int)[0]
    channel_idx = tf.tile(tf.range(8)[tf.newaxis, :], [bsz, 1])
    gather_idx = tf.stack([channel_idx, y_true_int], axis=-1)
    sw = tf.gather_nd(class_weights_tf, gather_idx)
    y_true_onehot = tf.one_hot(y_true_int, depth=6)
    ce = -tf.reduce_sum(y_true_onehot * tf.math.log(y_pred + 1e-7), axis=-1)
    return tf.reduce_mean(ce * sw)

def diagnosis_accuracy(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    y_pred_class = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
    return tf.reduce_mean(tf.cast(tf.equal(y_true_int, y_pred_class), tf.float32))

def nonzero_severity_mse(y_true, y_pred):
    mask = tf.cast(y_true > 0, tf.float32)
    sq_err = tf.square(y_pred - y_true) * mask
    return tf.reduce_sum(sq_err) / (tf.reduce_sum(mask) + 1e-7)

print("Loss/metric functions defined.")

Loss/metric functions defined.


2026-08-30 06:51:42.651904: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-08-30 06:51:42.651935: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-08-30 06:51:42.651942: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-08-30 06:51:42.651961: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-30 06:51:42.651989: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## Step 1: Build datasets with the masked RUL sample_weight

In [4]:
def to_multi_output_with_mask(inputs, labels):
    x_dict = {"x": inputs["x"], "x_rul_aux": inputs["x_rul_aux"]}
    y_dict = {"y_detection": labels["y_detection"], "y_diagnosis": labels["y_diagnosis"],
              "y_severity": labels["y_severity"], "y_rul_hours": labels["y_rul_hours"]}
    ones = tf.ones_like(labels["y_detection"])
    sw_dict = {"y_detection": ones, "y_diagnosis": ones, "y_severity": ones,
               "y_rul_hours": tf.cast(labels["rul_valid"], tf.float32)}
    return x_dict, y_dict, sw_dict

train_ds = make_dataset(INDEX_PATH, DATA_DIR, "train", SCALER_PATH, batch_size=BATCH_SIZE).map(to_multi_output_with_mask)
val_ds   = make_dataset(INDEX_PATH, DATA_DIR, "val",   SCALER_PATH, batch_size=BATCH_SIZE, shuffle_buffer=0).map(to_multi_output_with_mask)
print("Datasets built.")

Datasets built.


## Step 2: Define the HP-searchable model, search, then train the final model
Detection/Diagnosis/Severity load from Phase 4 and stay frozen in every trial and the final run - zero shared weights with RUL's LSTM branch, so those three heads are architecturally protected regardless of what the search finds.

In [5]:
import keras_tuner as kt

def build_hp_model(hp):
    units = hp.Choice("rul_units", [16, 32, 64])
    num_layers = hp.Choice("rul_num_layers", [1, 2, 3])
    dropout = hp.Choice("rul_dropout", [0.05, 0.1, 0.2])
    lr = hp.Choice("learning_rate", [1e-4, 3e-4, 1e-3])

    x_input = build_encoder_input()
    aux_input = keras.Input(shape=(2,), name="x_rul_aux")
    enc_out = build_encoder(x_input)
    det_out = build_detection_head(enc_out)
    diag_out = build_diagnosis_head(enc_out)
    sev_out = build_severity_head(enc_out, diag_out)
    lstm_out = build_rul_lstm_encoder(x_input, units=units, num_layers=num_layers, dropout=dropout)
    rul_out = build_rul_head(lstm_out, aux_input)

    model = keras.Model(inputs=[x_input, aux_input],
        outputs={"y_detection": det_out, "y_diagnosis": diag_out, "y_severity": sev_out, "y_rul_hours": rul_out})

    phase4_model = keras.models.load_model(PHASE4_CKPT, safe_mode=False, compile=False)
    phase4_layers = {layer.name: layer for layer in phase4_model.layers}
    for layer in model.layers:
        if layer.name in phase4_layers:
            try:
                layer.set_weights(phase4_layers[layer.name].get_weights())
            except ValueError:
                pass

    for layer in model.layers:
        if layer.name.startswith("rul_") or layer.name == "x_rul_aux":
            layer.trainable = True
        else:
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss={"y_detection": keras.losses.BinaryCrossentropy(), "y_diagnosis": weighted_diagnosis_loss,
              "y_severity": keras.losses.MeanSquaredError(), "y_rul_hours": keras.losses.MeanSquaredError()},
        loss_weights={"y_detection": 1.0, "y_diagnosis": 50.0, "y_severity": 20.0, "y_rul_hours": 1.0},
        metrics={"y_detection": keras.metrics.BinaryAccuracy(name="acc"), "y_diagnosis": diagnosis_accuracy,
                 "y_severity": nonzero_severity_mse, "y_rul_hours": keras.metrics.MeanAbsoluteError(name="mae")},
    )
    return model

print("HP model builder defined.")

HP model builder defined.


## Hyperparameter search (Hyperband)
Searches rul_units, rul_num_layers, rul_dropout, learning_rate. Uses REDUCED steps_per_epoch (1000 train / 200 val) during search only - full-size steps for every trial would make the search itself take hours. LSTM steps are naturally slower than TCN's, so this search may take longer than the TCN notebook's equivalent search - worth expecting, not a sign something is wrong.

## Final training run using the best found hyperparameters
Full-size steps_per_epoch this time. Same checkpoint safeguard as the TCN notebook - temp file, only promoted if it genuinely beats the recorded best.

In [6]:
best_units = best_hp.get("rul_units")
best_num_layers = best_hp.get("rul_num_layers")
best_dropout = best_hp.get("rul_dropout")
best_lr = best_hp.get("learning_rate")

x_input = build_encoder_input()
aux_input = keras.Input(shape=(2,), name="x_rul_aux")
enc_out = build_encoder(x_input)
det_out = build_detection_head(enc_out)
diag_out = build_diagnosis_head(enc_out)
sev_out = build_severity_head(enc_out, diag_out)
lstm_out = build_rul_lstm_encoder(x_input, units=best_units, num_layers=best_num_layers, dropout=best_dropout)
rul_out = build_rul_head(lstm_out, aux_input)

model = keras.Model(inputs=[x_input, aux_input],
    outputs={"y_detection": det_out, "y_diagnosis": diag_out, "y_severity": sev_out, "y_rul_hours": rul_out},
    name="phase5_rul_lstm_final")

phase4_model = keras.models.load_model(PHASE4_CKPT, safe_mode=False, compile=False)
phase4_layers = {layer.name: layer for layer in phase4_model.layers}
n_matched = 0
for layer in model.layers:
    if layer.name in phase4_layers:
        try:
            layer.set_weights(phase4_layers[layer.name].get_weights())
            n_matched += 1
        except ValueError as e:
            print(f"  skipped {layer.name}: {e}")
print(f"Warm-started {n_matched} matching layers from Phase 4 checkpoint.")

for layer in model.layers:
    if layer.name.startswith("rul_") or layer.name == "x_rul_aux":
        layer.trainable = True
    else:
        layer.trainable = False
print(f"Trainable variables: {len(model.trainable_variables)}, non-trainable: {len(model.non_trainable_variables)}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_lr, clipnorm=1.0),
    loss={"y_detection": keras.losses.BinaryCrossentropy(), "y_diagnosis": weighted_diagnosis_loss,
          "y_severity": keras.losses.MeanSquaredError(), "y_rul_hours": keras.losses.MeanSquaredError()},
    loss_weights={"y_detection": 1.0, "y_diagnosis": 50.0, "y_severity": 20.0, "y_rul_hours": 1.0},
    metrics={"y_detection": keras.metrics.BinaryAccuracy(name="acc"), "y_diagnosis": diagnosis_accuracy,
             "y_severity": nonzero_severity_mse, "y_rul_hours": keras.metrics.MeanAbsoluteError(name="mae")},
)

callbacks = [
    keras.callbacks.ModelCheckpoint(TEMP_CKPT, monitor="val_y_rul_hours_mae", mode="min", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_y_rul_hours_mae", mode="min", patience=4, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_y_rul_hours_mae", mode="min", factor=0.5, patience=2, verbose=1),
]

history = model.fit(train_ds, steps_per_epoch=STEPS_PER_EPOCH, validation_data=val_ds,
    validation_steps=VALIDATION_STEPS, epochs=EPOCHS, callbacks=callbacks, verbose=1)

NameError: name 'best_hp' is not defined

## Step 3: Promote the checkpoint ONLY if it genuinely beats the recorded best

In [ ]:
h = history.history
new_best_mae = min(h["val_y_rul_hours_mae"])
print("Best val_y_detection_acc:", max(h["val_y_detection_acc"]))
print("Best val_y_diagnosis_diagnosis_accuracy:", max(h["val_y_diagnosis_diagnosis_accuracy"]))
print("Best val_y_severity_nonzero_severity_mse:", min(h["val_y_severity_nonzero_severity_mse"]))
print("Best val_y_rul_hours_mae (this run):", new_best_mae)

old_best_mae = float("inf")
if os.path.exists(BEST_MAE_RECORD):
    with open(BEST_MAE_RECORD) as f:
        old_best_mae = json.load(f).get("best_val_rul_hours_mae", float("inf"))
print()
print("Previously recorded best MAE:", old_best_mae)
print("This run's best MAE:", new_best_mae)

if new_best_mae < old_best_mae:
    os.replace(TEMP_CKPT, PHASE5_CKPT)
    with open(BEST_MAE_RECORD, "w") as f:
        json.dump({"best_val_rul_hours_mae": new_best_mae}, f)
    print(">>> NEW BEST - promoted to", PHASE5_CKPT)
else:
    print(">>> Did NOT beat previous best - existing checkpoint left untouched, temp file discarded")
    if os.path.exists(TEMP_CKPT):
        os.remove(TEMP_CKPT)

## Test evaluation - run ONCE, when satisfied with val numbers

In [7]:
test_ds = make_dataset(INDEX_PATH, DATA_DIR, "test", SCALER_PATH, batch_size=BATCH_SIZE, shuffle_buffer=0).map(to_multi_output_with_mask)

loaded_model = keras.models.load_model(PHASE5_CKPT, safe_mode=False, custom_objects={
    "weighted_diagnosis_loss": weighted_diagnosis_loss, "diagnosis_accuracy": diagnosis_accuracy,
    "nonzero_severity_mse": nonzero_severity_mse})
results = loaded_model.evaluate(test_ds, steps=TEST_STEPS, verbose=1, return_dict=True)
print()
for k, v in results.items():
    print(f"{k}: {v}")

2026-08-30 07:07:59.021519: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


522/522 ━━━━━━━━━━━━━━━━━━━━ 33s 58ms/step - loss: 0.2698 - y_detection_acc: 0.9837 - y_detection_loss: 0.0605 - y_diagnosis_diagnosis_accuracy: 0.9906 - y_diagnosis_loss: 3.7329e-04 - y_rul_hours_loss: 0.1688 - y_rul_hours_mae: 1.0674 - y_severity_loss: 0.0011 - y_severity_nonzero_severity_mse: 0.0054

loss: 0.26984092593193054
y_detection_acc: 0.9836865663528442
y_detection_loss: 0.06048225238919258
y_diagnosis_diagnosis_accuracy: 0.9905654788017273
y_diagnosis_loss: 0.0003732923069037497
y_rul_hours_loss: 0.1688387244939804
y_rul_hours_mae: 1.0674487352371216
y_severity_loss: 0.0010927696712315083
y_severity_nonzero_severity_mse: 0.005395513027906418


In [1]:
import sys, os, json, warnings, itertools, time
warnings.filterwarnings("ignore")
sys.path.insert(0, "/Users/harsh/Documents/UAV_Engine/validation")
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tf_data_pipeline import make_dataset
from model_architectures import (build_encoder_input, build_encoder, build_detection_head,
                                   build_diagnosis_head, build_severity_head, build_rul_head,
                                   build_rul_tcn_encoder)

tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

DATA_DIR = "/Users/harsh/Documents/UAV_Engine/validation/chunks"
INDEX_PATH = os.path.join(DATA_DIR, "scenario_index.json")
SCALER_PATH = "/Users/harsh/Documents/UAV_Engine/validation/models/scaler.pkl"
PHASE4_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase4_severity.keras"
PHASE5_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp.keras"
TEMP_CKPT = PHASE5_CKPT.replace(".keras", "_temp.keras")
BEST_MAE_RECORD = "/Users/harsh/Documents/UAV_Engine/validation/models/phase5_manual_hp_best_mae.json"
CLASS_WEIGHTS_PATH = "/Users/harsh/Documents/UAV_Engine/validation/models/diag_class_weights_keras.npy"

BATCH_SIZE = 128
SEARCH_STEPS_PER_EPOCH = 800
SEARCH_VALIDATION_STEPS = 150
SEARCH_EPOCHS = 3
FINAL_STEPS_PER_EPOCH = 4196
FINAL_VALIDATION_STEPS = 522
FINAL_TEST_STEPS = 522
FINAL_EPOCHS = 12

class_weights_np = np.load(CLASS_WEIGHTS_PATH)
class_weights_tf = tf.constant(class_weights_np, dtype=tf.float32)

def weighted_diagnosis_loss(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    bsz = tf.shape(y_true_int)[0]
    channel_idx = tf.tile(tf.range(8)[tf.newaxis, :], [bsz, 1])
    gather_idx = tf.stack([channel_idx, y_true_int], axis=-1)
    sw = tf.gather_nd(class_weights_tf, gather_idx)
    y_true_onehot = tf.one_hot(y_true_int, depth=6)
    ce = -tf.reduce_sum(y_true_onehot * tf.math.log(y_pred + 1e-7), axis=-1)
    return tf.reduce_mean(ce * sw)

def diagnosis_accuracy(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    y_pred_class = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
    return tf.reduce_mean(tf.cast(tf.equal(y_true_int, y_pred_class), tf.float32))

def nonzero_severity_mse(y_true, y_pred):
    mask = tf.cast(y_true > 0, tf.float32)
    sq_err = tf.square(y_pred - y_true) * mask
    return tf.reduce_sum(sq_err) / (tf.reduce_sum(mask) + 1e-7)

def to_multi_output_with_mask(inputs, labels):
    x_dict = {"x": inputs["x"], "x_rul_aux": inputs["x_rul_aux"]}
    y_dict = {"y_detection": labels["y_detection"], "y_diagnosis": labels["y_diagnosis"],
              "y_severity": labels["y_severity"], "y_rul_hours": labels["y_rul_hours"]}
    ones = tf.ones_like(labels["y_detection"])
    sw_dict = {"y_detection": ones, "y_diagnosis": ones, "y_severity": ones,
               "y_rul_hours": tf.cast(labels["rul_valid"], tf.float32)}
    return x_dict, y_dict, sw_dict

print("Datasets will be built fresh per trial to avoid any dataset-state reuse issues.")
def build_model(channels, num_layers, dropout, lr, optimizer_name):
    x_input = build_encoder_input()
    aux_input = keras.Input(shape=(6,), name="x_rul_aux")
    enc_out = build_encoder(x_input)
    det_out = build_detection_head(enc_out)
    diag_out = build_diagnosis_head(enc_out)
    sev_out = build_severity_head(enc_out, diag_out)
    rul_tcn_out = build_rul_tcn_encoder(x_input, channels=channels, num_layers=num_layers, dropout=dropout)
    rul_out = build_rul_head(rul_tcn_out, aux_input)

    model = keras.Model(inputs=[x_input, aux_input],
        outputs={"y_detection": det_out, "y_diagnosis": diag_out, "y_severity": sev_out, "y_rul_hours": rul_out})

    phase4_model = keras.models.load_model(PHASE4_CKPT, safe_mode=False, compile=False)
    phase4_layers = {layer.name: layer for layer in phase4_model.layers}
    for layer in model.layers:
        if layer.name in phase4_layers:
            try:
                layer.set_weights(phase4_layers[layer.name].get_weights())
            except ValueError:
                pass

    for layer in model.layers:
        if layer.name.startswith("rul_") or layer.name == "x_rul_aux":
            layer.trainable = True
        else:
            layer.trainable = False

    if optimizer_name == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0)
    elif optimizer_name == "adamw":
        opt = keras.optimizers.AdamW(learning_rate=lr, clipnorm=1.0)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=lr, clipnorm=1.0)

    model.compile(
        optimizer=opt,
        loss={"y_detection": keras.losses.BinaryCrossentropy(), "y_diagnosis": weighted_diagnosis_loss,
              "y_severity": keras.losses.MeanSquaredError(), "y_rul_hours": keras.losses.MeanSquaredError()},
        loss_weights={"y_detection": 1.0, "y_diagnosis": 50.0, "y_severity": 20.0, "y_rul_hours": 1.0},
        metrics={"y_detection": keras.metrics.BinaryAccuracy(name="acc"), "y_diagnosis": diagnosis_accuracy,
                 "y_severity": nonzero_severity_mse, "y_rul_hours": keras.metrics.MeanAbsoluteError(name="mae")},
    )
    return model

# Curated set of combinations to try - not exhaustive, but covers the key dimensions
# we have reason to think matter based on everything learned so far.
candidates = [
    {"channels": 32, "num_layers": 6, "dropout": 0.1, "lr": 1e-3, "optimizer": "adam"},
    {"channels": 64, "num_layers": 6, "dropout": 0.1, "lr": 1e-3, "optimizer": "adam"},
    {"channels": 32, "num_layers": 4, "dropout": 0.1, "lr": 1e-3, "optimizer": "adam"},
    {"channels": 32, "num_layers": 6, "dropout": 0.2, "lr": 1e-3, "optimizer": "adam"},
    {"channels": 32, "num_layers": 6, "dropout": 0.1, "lr": 3e-4, "optimizer": "adam"},
    {"channels": 32, "num_layers": 6, "dropout": 0.1, "lr": 1e-3, "optimizer": "adamw"},
    {"channels": 32, "num_layers": 6, "dropout": 0.1, "lr": 1e-3, "optimizer": "rmsprop"},
]

results = []
for i, cand in enumerate(candidates):
    print()
    print(f"=== Trial {i+1}/{len(candidates)}: {cand} ===")
    train_ds = make_dataset(INDEX_PATH, DATA_DIR, "train", SCALER_PATH, batch_size=BATCH_SIZE).map(to_multi_output_with_mask)
    val_ds   = make_dataset(INDEX_PATH, DATA_DIR, "val",   SCALER_PATH, batch_size=BATCH_SIZE, shuffle_buffer=0).map(to_multi_output_with_mask)
    model = build_model(cand["channels"], cand["num_layers"], cand["dropout"], cand["lr"], cand["optimizer"])
    t0 = time.time()
    hist = model.fit(train_ds, steps_per_epoch=SEARCH_STEPS_PER_EPOCH, validation_data=val_ds,
        validation_steps=SEARCH_VALIDATION_STEPS, epochs=SEARCH_EPOCHS, verbose=1)
    elapsed = time.time() - t0
    best_mae = min(hist.history["val_y_rul_hours_mae"])
    print(f"  best val_rul_mae={best_mae:.4f}  (took {elapsed:.0f}s)")
    results.append({**cand, "best_mae": best_mae})

results.sort(key=lambda r: r["best_mae"])
print()
print("=== ALL TRIALS, sorted best to worst ===")
for r in results:
    print(r)

best = results[0]
print()
print("WINNER:", best)
# --- Final training with the winning combination, full-size steps ---
print()
print("Starting FINAL training with winning hyperparameters...")
train_ds = make_dataset(INDEX_PATH, DATA_DIR, "train", SCALER_PATH, batch_size=BATCH_SIZE).map(to_multi_output_with_mask)
val_ds   = make_dataset(INDEX_PATH, DATA_DIR, "val",   SCALER_PATH, batch_size=BATCH_SIZE, shuffle_buffer=0).map(to_multi_output_with_mask)

model = build_model(best["channels"], best["num_layers"], best["dropout"], best["lr"], best["optimizer"])
print(f"Trainable: {len(model.trainable_variables)}, frozen: {len(model.non_trainable_variables)}")

callbacks = [
    keras.callbacks.ModelCheckpoint(TEMP_CKPT, monitor="val_y_rul_hours_mae", mode="min", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_y_rul_hours_mae", mode="min", patience=4, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_y_rul_hours_mae", mode="min", factor=0.5, patience=2, verbose=1),
]

history = model.fit(train_ds, steps_per_epoch=FINAL_STEPS_PER_EPOCH, validation_data=val_ds,
    validation_steps=FINAL_VALIDATION_STEPS, epochs=FINAL_EPOCHS, callbacks=callbacks, verbose=1)

h = history.history
new_best_mae = min(h["val_y_rul_hours_mae"])
print()
print("Best val_y_detection_acc:", max(h["val_y_detection_acc"]))
print("Best val_y_diagnosis_diagnosis_accuracy:", max(h["val_y_diagnosis_diagnosis_accuracy"]))
print("Best val_y_severity_nonzero_severity_mse:", min(h["val_y_severity_nonzero_severity_mse"]))
print("Best val_y_rul_hours_mae (final run):", new_best_mae)

old_best_mae = float("inf")
if os.path.exists(BEST_MAE_RECORD):
    with open(BEST_MAE_RECORD) as f:
        old_best_mae = json.load(f).get("best_val_rul_hours_mae", float("inf"))
print()
print("Previously recorded best MAE:", old_best_mae)
print("This run best MAE:", new_best_mae)

if new_best_mae < old_best_mae:
    os.replace(TEMP_CKPT, PHASE5_CKPT)
    with open(BEST_MAE_RECORD, "w") as f:
        json.dump({"best_val_rul_hours_mae": new_best_mae, **best}, f)
    print(">>> NEW BEST - promoted to", PHASE5_CKPT)
else:
    print(">>> Did NOT beat previous best")
    if os.path.exists(TEMP_CKPT):
        os.remove(TEMP_CKPT)

print()
print("=== TEST EVALUATION ===")
test_ds = make_dataset(INDEX_PATH, DATA_DIR, "test", SCALER_PATH, batch_size=BATCH_SIZE, shuffle_buffer=0).map(to_multi_output_with_mask)
loaded_model = keras.models.load_model(PHASE5_CKPT, safe_mode=False, custom_objects={
    "weighted_diagnosis_loss": weighted_diagnosis_loss, "diagnosis_accuracy": diagnosis_accuracy,
    "nonzero_severity_mse": nonzero_severity_mse})
results_test = loaded_model.evaluate(test_ds, steps=FINAL_TEST_STEPS, verbose=1, return_dict=True)
print()
for k, v in results_test.items():
    print(f"{k}: {v}")


TensorFlow: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Datasets will be built fresh per trial to avoid any dataset-state reuse issues.

=== Trial 1/7: {'channels': 32, 'num_layers': 6, 'dropout': 0.1, 'lr': 0.001, 'optimizer': 'adam'} ===


2026-08-30 07:25:15.872508: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-08-30 07:25:15.872783: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-08-30 07:25:15.872827: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-08-30 07:25:15.872883: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-30 07:25:15.872904: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/3


2026-08-30 07:25:20.179886: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


 96/800 ━━━━━━━━━━━━━━━━━━━━ 4:12 359ms/step - loss: 1.5853 - y_detection_acc: 0.9796 - y_detection_loss: 0.0737 - y_diagnosis_diagnosis_accuracy: 0.9877 - y_diagnosis_loss: 4.9418e-04 - y_rul_hours_loss: 1.4562 - y_rul_hours_mae: 1.5725 - y_severity_loss: 0.0015 - y_severity_nonzero_severity_mse: 0.0102

KeyboardInterrupt: 

In [ ]:
import sys
import os
import json
import warnings
import time

warnings.filterwarnings("ignore")
sys.path.insert(0, "/Users/harsh/Documents/UAV_Engine/validation")

import numpy as np
import tensorflow as tf
from tensorflow import keras

from tf_data_pipeline import make_dataset
from model_architectures import (
    build_encoder_input,
    build_encoder,
    build_detection_head,
    build_diagnosis_head,
    build_severity_head,
    build_rul_head,
    build_rul_lstm_encoder,
)

# ============================================================
# FINAL RUL TRAINING - LSTM
# No hyperparameter search - one final run
# ============================================================

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"), flush=True)

DATA_DIR = "/Users/harsh/Documents/UAV_Engine/validation/chunks"
INDEX_PATH = os.path.join(DATA_DIR, "scenario_index.json")
SCALER_PATH = "/Users/harsh/Documents/UAV_Engine/validation/models/scaler.pkl"

PHASE4_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase4_severity.keras"
PHASE5_CKPT = "/Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp.keras"
TEMP_CKPT = PHASE5_CKPT.replace(".keras", "_temp.keras")
BEST_MAE_RECORD = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "phase5_manual_hp_best_mae.json"
)
CLASS_WEIGHTS_PATH = (
    "/Users/harsh/Documents/UAV_Engine/validation/models/"
    "diag_class_weights_keras.npy"
)

# ============================================================
# FINAL PARAMETERS
# ============================================================

BATCH_SIZE = 128

LSTM_UNITS = 32
LSTM_LAYERS = 2
DROPOUT = 0.10
LEARNING_RATE = 3e-4
OPTIMIZER_NAME = "adam"

FINAL_STEPS_PER_EPOCH = 4196
FINAL_VALIDATION_STEPS = 522
FINAL_TEST_STEPS = 522
FINAL_EPOCHS = 12

print("\n=== FINAL LSTM HYPERPARAMETERS ===", flush=True)
print("LSTM units   :", LSTM_UNITS)
print("LSTM layers  :", LSTM_LAYERS)
print("dropout      :", DROPOUT)
print("learning_rate:", LEARNING_RATE)
print("optimizer    :", OPTIMIZER_NAME)
print("batch_size   :", BATCH_SIZE)
print("epochs       :", FINAL_EPOCHS)
print("================================\n", flush=True)

# ============================================================
# LOSSES / METRICS
# ============================================================

class_weights_np = np.load(CLASS_WEIGHTS_PATH)
class_weights_tf = tf.constant(class_weights_np, dtype=tf.float32)


def weighted_diagnosis_loss(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    bsz = tf.shape(y_true_int)[0]

    channel_idx = tf.tile(
        tf.range(8)[tf.newaxis, :],
        [bsz, 1]
    )

    gather_idx = tf.stack(
        [channel_idx, y_true_int],
        axis=-1
    )

    sw = tf.gather_nd(class_weights_tf, gather_idx)

    y_true_onehot = tf.one_hot(
        y_true_int,
        depth=6
    )

    ce = -tf.reduce_sum(
        y_true_onehot * tf.math.log(y_pred + 1e-7),
        axis=-1
    )

    return tf.reduce_mean(ce * sw)


def diagnosis_accuracy(y_true, y_pred):
    y_true_int = tf.cast(y_true, tf.int32)
    y_pred_class = tf.cast(
        tf.argmax(y_pred, axis=-1),
        tf.int32
    )

    return tf.reduce_mean(
        tf.cast(
            tf.equal(y_true_int, y_pred_class),
            tf.float32
        )
    )


def nonzero_severity_mse(y_true, y_pred):
    mask = tf.cast(y_true > 0, tf.float32)

    sq_err = tf.square(
        y_pred - y_true
    ) * mask

    return tf.reduce_sum(sq_err) / (
        tf.reduce_sum(mask) + 1e-7
    )


# ============================================================
# DATASET ADAPTER
# ============================================================

def to_multi_output_with_mask(inputs, labels):
    x_dict = {
        "x": inputs["x"],
        "x_rul_aux": inputs["x_rul_aux"],
    }

    y_dict = {
        "y_detection": labels["y_detection"],
        "y_diagnosis": labels["y_diagnosis"],
        "y_severity": labels["y_severity"],
        "y_rul_hours": labels["y_rul_hours"],
    }

    ones = tf.ones_like(
        labels["y_detection"]
    )

    sw_dict = {
        "y_detection": ones,
        "y_diagnosis": ones,
        "y_severity": ones,
        "y_rul_hours": tf.cast(
            labels["rul_valid"],
            tf.float32
        ),
    }

    return x_dict, y_dict, sw_dict


# ============================================================
# MODEL
# ============================================================

def build_final_model():
    print("Building final model...", flush=True)

    x_input = build_encoder_input()

    aux_input = keras.Input(
        shape=(6,),
        name="x_rul_aux"
    )

    enc_out = build_encoder(x_input)

    det_out = build_detection_head(enc_out)
    diag_out = build_diagnosis_head(enc_out)
    sev_out = build_severity_head(
        enc_out,
        diag_out
    )

    # Independent LSTM branch for RUL.
    # Detection/Diagnosis/Severity continue using the frozen TCN path.
    rul_lstm_out = build_rul_lstm_encoder(
        x_input,
        units=LSTM_UNITS,
        num_layers=LSTM_LAYERS,
        dropout=DROPOUT,
    )

    rul_out = build_rul_head(
        rul_lstm_out,
        aux_input
    )

    model = keras.Model(
        inputs=[x_input, aux_input],
        outputs={
            "y_detection": det_out,
            "y_diagnosis": diag_out,
            "y_severity": sev_out,
            "y_rul_hours": rul_out,
        },
    )

    # --------------------------------------------------------
    # Load pretrained Phase 4 weights
    # --------------------------------------------------------

    print(
        "Loading pretrained Phase 4 weights...",
        flush=True
    )

    phase4_model = keras.models.load_model(
        PHASE4_CKPT,
        safe_mode=False,
        compile=False,
    )

    phase4_layers = {
        layer.name: layer
        for layer in phase4_model.layers
    }

    copied = 0

    for layer in model.layers:
        if layer.name in phase4_layers:
            try:
                old_weights = phase4_layers[
                    layer.name
                ].get_weights()

                new_weights = layer.get_weights()

                if old_weights and new_weights:
                    layer.set_weights(old_weights)
                    copied += 1

            except (ValueError, RuntimeError):
                pass

    print(
        f"Copied pretrained weights for {copied} layers.",
        flush=True
    )

    # --------------------------------------------------------
    # Freeze everything except RUL branch
    # --------------------------------------------------------

    for layer in model.layers:
        if layer.name.startswith("rul_"):
            layer.trainable = True
        else:
            layer.trainable = False

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0,
    )

    # --------------------------------------------------------
    # Compile
    # --------------------------------------------------------

    model.compile(
        optimizer=optimizer,

        loss={
            "y_detection":
                keras.losses.BinaryCrossentropy(),

            "y_diagnosis":
                weighted_diagnosis_loss,

            "y_severity":
                keras.losses.MeanSquaredError(),

            "y_rul_hours":
                keras.losses.MeanSquaredError(),
        },

        loss_weights={
            "y_detection": 1.0,
            "y_diagnosis": 50.0,
            "y_severity": 20.0,
            "y_rul_hours": 1.0,
        },

        metrics={
            "y_detection":
                keras.metrics.BinaryAccuracy(
                    name="acc"
                ),

            "y_diagnosis":
                diagnosis_accuracy,

            "y_severity":
                nonzero_severity_mse,

            "y_rul_hours":
                keras.metrics.MeanAbsoluteError(
                    name="mae"
                ),
        },
    )

    print(
        f"Trainable variables: "
        f"{len(model.trainable_variables)}",
        flush=True
    )

    return model


# ============================================================
# MAIN TRAINING
# ============================================================

print("Creating training dataset...", flush=True)

train_ds = make_dataset(
    INDEX_PATH,
    DATA_DIR,
    "train",
    SCALER_PATH,
    batch_size=BATCH_SIZE,
).map(
    to_multi_output_with_mask
)

print("Creating validation dataset...", flush=True)

val_ds = make_dataset(
    INDEX_PATH,
    DATA_DIR,
    "val",
    SCALER_PATH,
    batch_size=BATCH_SIZE,
    shuffle_buffer=0,
).map(
    to_multi_output_with_mask
)

model = build_final_model()

# Remove stale temporary checkpoint
if os.path.exists(TEMP_CKPT):
    os.remove(TEMP_CKPT)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        TEMP_CKPT,
        monitor="val_y_rul_hours_mae",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_y_rul_hours_mae",
        mode="min",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_y_rul_hours_mae",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

print("\n========================================", flush=True)
print("STARTING FINAL LSTM RUL TRAINING", flush=True)
print("========================================\n", flush=True)

t0 = time.time()

history = model.fit(
    train_ds,
    steps_per_epoch=FINAL_STEPS_PER_EPOCH,

    validation_data=val_ds,
    validation_steps=FINAL_VALIDATION_STEPS,

    epochs=FINAL_EPOCHS,

    callbacks=callbacks,

    # IMPORTANT: show epoch/step progress
    verbose=1,
)

elapsed = time.time() - t0

print(
    f"\nFinal training finished in "
    f"{elapsed / 60:.1f} minutes.",
    flush=True
)

# ============================================================
# TRAINING RESULTS
# ============================================================

h = history.history

new_best_mae = min(
    h["val_y_rul_hours_mae"]
)

print("\n========================================")
print("FINAL TRAINING RESULTS")
print("========================================")

print(
    "Best val detection accuracy:",
    max(h["val_y_detection_acc"])
)

print(
    "Best val diagnosis accuracy:",
    max(
        h[
            "val_y_diagnosis_diagnosis_accuracy"
        ]
    )
)

print(
    "Best val severity MSE:",
    min(
        h[
            "val_y_severity_nonzero_severity_mse"
        ]
    )
)

print(
    "Best val RUL MAE:",
    new_best_mae
)

print("========================================\n")

# ============================================================
# PROMOTE BEST CHECKPOINT
# ============================================================

old_best_mae = float("inf")

if os.path.exists(BEST_MAE_RECORD):
    try:
        with open(BEST_MAE_RECORD) as f:
            old_best_mae = json.load(f).get(
                "best_val_rul_hours_mae",
                float("inf")
            )
    except Exception:
        old_best_mae = float("inf")

print(
    "Previously recorded best MAE:",
    old_best_mae
)

print(
    "This run best MAE:",
    new_best_mae
)

if os.path.exists(TEMP_CKPT):

    # Since this is explicitly the final run, promote it
    # if it beats the previous recorded best.
    if new_best_mae < old_best_mae:
        os.replace(
            TEMP_CKPT,
            PHASE5_CKPT
        )

        with open(
            BEST_MAE_RECORD,
            "w"
        ) as f:
            json.dump(
                {
                    "best_val_rul_hours_mae":
                        new_best_mae,
                    "lstm_units":
                        LSTM_UNITS,
                    "lstm_layers":
                        LSTM_LAYERS,
                    "dropout":
                        DROPOUT,
                    "lr":
                        LEARNING_RATE,
                    "optimizer":
                        OPTIMIZER_NAME,
                },
                f,
                indent=2,
            )

        print(
            "\n>>> NEW BEST CHECKPOINT SAVED:",
            PHASE5_CKPT,
            flush=True
        )

    else:
        # Keep the temporary best checkpoint as a backup
        # rather than deleting it automatically.
        backup_path = PHASE5_CKPT.replace(
            ".keras",
            "_final_run.keras"
        )

        if os.path.exists(backup_path):
            os.remove(backup_path)

        os.replace(
            TEMP_CKPT,
            backup_path
        )

        print(
            "\n>>> Previous checkpoint is still better.",
            flush=True
        )
        print(
            "Final-run checkpoint saved to:",
            backup_path,
            flush=True
        )

else:
    print(
        "\nWARNING: Temporary best checkpoint was not created.",
        flush=True
    )

# ============================================================
# TEST EVALUATION
# ============================================================

print("\n========================================")
print("TEST EVALUATION")
print("========================================")

if not os.path.exists(PHASE5_CKPT):
    raise FileNotFoundError(
        f"Best checkpoint not found: {PHASE5_CKPT}"
    )

print(
    "Loading:",
    PHASE5_CKPT,
    flush=True
)

loaded_model = keras.models.load_model(
    PHASE5_CKPT,
    safe_mode=False,
    custom_objects={
        "weighted_diagnosis_loss":
            weighted_diagnosis_loss,

        "diagnosis_accuracy":
            diagnosis_accuracy,

        "nonzero_severity_mse":
            nonzero_severity_mse,
    },
)

print("Creating test dataset...", flush=True)

test_ds = make_dataset(
    INDEX_PATH,
    DATA_DIR,
    "test",
    SCALER_PATH,
    batch_size=BATCH_SIZE,
    shuffle_buffer=0,
).map(
    to_multi_output_with_mask
)

print("Running test evaluation...", flush=True)

results_test = loaded_model.evaluate(
    test_ds,
    steps=FINAL_TEST_STEPS,
    verbose=1,
    return_dict=True,
)

print("\n========================================")
print("FINAL TEST RESULTS")
print("========================================")

for key, value in results_test.items():
    print(
        f"{key}: {value}"
    )

print("========================================")
print("DONE")
print("========================================")


TensorFlow: 2.16.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

=== FINAL LSTM HYPERPARAMETERS ===
LSTM units   : 32
LSTM layers  : 2
dropout      : 0.1
learning_rate: 0.0003
optimizer    : adam
batch_size   : 128
epochs       : 12

Creating training dataset...


2026-08-30 07:35:29.101934: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-08-30 07:35:29.101975: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-08-30 07:35:29.102000: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-08-30 07:35:29.102033: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-30 07:35:29.102056: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Creating validation dataset...
Building final model...
Loading pretrained Phase 4 weights...
Copied pretrained weights for 19 layers.
Trainable variables: 12

STARTING FINAL LSTM RUL TRAINING

Epoch 1/12


2026-08-30 07:35:33.479301: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


4196/4196 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - loss: 0.3550 - y_detection_acc: 0.9790 - y_detection_loss: 0.0774 - y_diagnosis_diagnosis_accuracy: 0.9884 - y_diagnosis_loss: 4.4680e-04 - y_rul_hours_loss: 0.2244 - y_rul_hours_mae: 1.0824 - y_severity_loss: 0.0015 - y_severity_nonzero_severity_mse: 0.0108
Epoch 1: val_y_rul_hours_mae improved from None to 0.97819, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_temp.keras

Epoch 1: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase5_rul_manual_hp_temp.keras
4196/4196 ━━━━━━━━━━━━━━━━━━━━ 861s 203ms/step - loss: 0.3550 - y_detection_acc: 0.9790 - y_detection_loss: 0.0774 - y_diagnosis_diagnosis_accuracy: 0.9884 - y_diagnosis_loss: 4.4680e-04 - y_rul_hours_loss: 0.2244 - y_rul_hours_mae: 1.0824 - y_severity_loss: 0.0015 - y_severity_nonzero_severity_mse: 0.0108 - val_loss: 0.2757 - val_y_detection_acc: 0.9843 - val_y_detection_loss: 0.0555 - val_y_diagnosis_diagnosis